In [1]:
import pandas as pd
from pathlib import Path

# Project paths
PROJECT_DIR = Path.cwd().parent
ORIGINAL_DIR = PROJECT_DIR / "datasets" / "original"
CLEANED_DIR = PROJECT_DIR / "datasets" / "cleaned"

CLEANED_DIR.mkdir(parents=True, exist_ok=True)

# Load original datasets
products = pd.read_csv(ORIGINAL_DIR / "products.csv")
locations = pd.read_csv(ORIGINAL_DIR / "warehouse_locations.csv")
order_lines = pd.read_csv(ORIGINAL_DIR / "order_lines.csv")
inventory = pd.read_csv(ORIGINAL_DIR / "inventory.csv")

print("Original datasets loaded successfully!")

Original datasets loaded successfully!


In [2]:
# Clean text columns

datasets = {
    "products": products,
    "locations": locations,
    "order_lines": order_lines,
    "inventory": inventory
}

for name, df in datasets.items():
    text_columns = df.select_dtypes(include=["object", "str"]).columns

    for column in text_columns:
        df[column] = df[column].str.strip()

    print(f"{name} text columns cleaned")

products text columns cleaned
locations text columns cleaned
order_lines text columns cleaned
inventory text columns cleaned


In [3]:
# Convert date column

order_lines["Order_Date"] = pd.to_datetime(
    order_lines["Order_Date"],
    format="%d-%m-%Y",
    errors="raise"
)

# Convert numeric columns

products["Required_Capacity"] = pd.to_numeric(
    products["Required_Capacity"], errors="raise"
)

for column in ["X", "Y", "Capacity"]:
    locations[column] = pd.to_numeric(
        locations[column], errors="raise"
    )

order_lines["Quantity"] = pd.to_numeric(
    order_lines["Quantity"], errors="raise"
)

for column in ["Current_Stock", "Minimum_Stock"]:
    inventory[column] = pd.to_numeric(
        inventory[column], errors="raise"
    )

# Standardize categorical values

locations["Status"] = locations["Status"].str.upper()
locations["Allowed_Category"] = locations["Allowed_Category"].str.upper()

print("Data type conversion completed!")

Data type conversion completed!


In [9]:
# Check invalid or negative values

print("Invalid product capacity:", (products["Required_Capacity"] <= 0).sum())

print("Invalid location capacity:", (locations["Capacity"] <= 0).sum())

print("Invalid order quantity:", (order_lines["Quantity"] <= 0).sum())

print("Invalid current stock:", (inventory["Current_Stock"] < 0).sum())

print("Invalid minimum stock:", (inventory["Minimum_Stock"] < 0).sum())

print("Invalid location status:", (~locations["Status"].isin(["EMPTY", "OCCUPIED"])).sum())

Invalid product capacity: 0
Invalid location capacity: 0
Invalid order quantity: 0
Invalid current stock: 0
Invalid minimum stock: 0
Invalid location status: 0


In [5]:
# Check missing values after cleaning

for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isnull().sum())

# Check duplicate primary keys

print("\nDuplicate IDs")

print("Products:", products["SKU_ID"].duplicated().sum())
print("Locations:", locations["Location_ID"].duplicated().sum())
print("Order Lines:", order_lines["Order_Line_ID"].duplicated().sum())
print("Inventory:", inventory["SKU_ID"].duplicated().sum())


products
SKU_ID               0
Product_Name         0
Category             0
Required_Capacity    0
Current_Location     0
dtype: int64

locations
Location_ID         0
Aisle               0
X                   0
Y                   0
Capacity            0
Status              0
Allowed_Category    0
dtype: int64

order_lines
Order_Line_ID    0
Order_ID         0
Order_Date       0
SKU_ID           0
Quantity         0
dtype: int64

inventory
SKU_ID           0
Current_Stock    0
Minimum_Stock    0
dtype: int64

Duplicate IDs
Products: 0
Locations: 0
Order Lines: 0
Inventory: 0


In [6]:
# Save cleaned datasets

products.to_csv(CLEANED_DIR / "products_clean.csv", index=False)

locations.to_csv(CLEANED_DIR / "warehouse_locations_clean.csv", index=False)

order_lines.to_csv(CLEANED_DIR / "order_lines_clean.csv", index=False)

inventory.to_csv(CLEANED_DIR / "inventory_clean.csv", index=False)

print("All cleaned datasets saved successfully!")

All cleaned datasets saved successfully!


In [7]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
CLEANED_DIR = PROJECT_DIR / "datasets" / "cleaned"

products = pd.read_csv(CLEANED_DIR / "products_clean.csv")
locations = pd.read_csv(CLEANED_DIR / "warehouse_locations_clean.csv")
order_lines = pd.read_csv(CLEANED_DIR / "order_lines_clean.csv")
inventory = pd.read_csv(CLEANED_DIR / "inventory_clean.csv")

print("Cleaned datasets loaded successfully!")

Cleaned datasets loaded successfully!


In [8]:
# Check whether all order SKUs exist in products

invalid_order_skus = set(order_lines["SKU_ID"]) - set(products["SKU_ID"])

# Check whether all inventory SKUs exist in products

invalid_inventory_skus = set(inventory["SKU_ID"]) - set(products["SKU_ID"])

print("Invalid SKUs in orders:", invalid_order_skus)
print("Invalid SKUs in inventory:", invalid_inventory_skus)

# Check whether product and inventory SKU sets match

print("Missing inventory SKUs:",
      set(products["SKU_ID"]) - set(inventory["SKU_ID"]))

Invalid SKUs in orders: set()
Invalid SKUs in inventory: set()
Missing inventory SKUs: set()


In [11]:
# Check whether current product locations exist

invalid_locations = set(products["Current_Location"]) - set(locations["Location_ID"])

print("Invalid product locations:", invalid_locations)

# Check whether multiple products share the same location

print("Products sharing a location:", products["Current_Location"].duplicated().sum())

Invalid product locations: set()
Products sharing a location: 0


In [12]:
# Identify locations assigned to products

assigned_locations = set(products["Current_Location"])

# Expected status based on product assignments

locations["Expected_Status"] = locations["Location_ID"].apply(
    lambda x: "OCCUPIED" if x in assigned_locations else "EMPTY"
)

# Compare actual and expected status

status_mismatch = locations[
    locations["Status"] != locations["Expected_Status"]
]

display(status_mismatch)

print("Status mismatches:", len(status_mismatch))

,Location_ID,Aisle,X,Y,Capacity,Status,Allowed_Category,Expected_Status


Status mismatches: 0


In [13]:
# Connect products with their current locations

product_location_check = products.merge(
    locations,
    left_on="Current_Location",
    right_on="Location_ID",
    how="left"
)

# Check capacity compatibility

capacity_issues = product_location_check[
    product_location_check["Required_Capacity"] >
    product_location_check["Capacity"]
]

# Check category compatibility

category_issues = product_location_check[
    (product_location_check["Allowed_Category"] != "ALL") &
    (product_location_check["Category"].str.upper() !=
     product_location_check["Allowed_Category"])
]

print("Capacity issues:", len(capacity_issues))
display(capacity_issues)

print("Category compatibility issues:", len(category_issues))
display(category_issues)

Capacity issues: 0


,SKU_ID,Product_Name,Category,Required_Capacity,Current_Location,Location_ID,Aisle,X,Y,Capacity,Status,Allowed_Category,Expected_Status


Category compatibility issues: 0


,SKU_ID,Product_Name,Category,Required_Capacity,Current_Location,Location_ID,Aisle,X,Y,Capacity,Status,Allowed_Category,Expected_Status


In [14]:
# Summarize validation results

validation_results = {
    "Invalid Order SKUs": len(invalid_order_skus),
    "Invalid Inventory SKUs": len(invalid_inventory_skus),
    "Invalid Product Locations": len(invalid_locations),
    "Shared Product Locations": products["Current_Location"].duplicated().sum(),
    "Location Status Mismatches": len(status_mismatch),
    "Capacity Issues": len(capacity_issues),
    "Category Issues": len(category_issues)
}

validation_summary = pd.DataFrame(
    list(validation_results.items()),
    columns=["Validation Check", "Issue Count"]
)

display(validation_summary)

,Validation Check,Issue Count
0,Invalid Order SKUs,0
1,Invalid Inventory SKUs,0
2,Invalid Product Locations,0
3,Shared Product Locations,0
4,Location Status Mismatches,0
5,Capacity Issues,0
6,Category Issues,0
